# CityFlo Bus Service — Metro Cities: Data Cleaning and Preparation

**Dataset:** `cityflo_bus_service_metro_cities.csv` (3,258 rows × 36 columns)
**Workflow followed:** *Exploratory Data Analysis — A Standard 23-Step Checklist for Any Dataset*
(Classroom Computer Institute — Data Analytics)

This notebook walks through **all 23 steps** of the checklist, in order, across four phases:

| Phase | Steps | Goal |
|---|---|---|
| Phase 1 — Inspect | 1–8 | Understand the shape, structure, and quality of the raw data |
| Phase 2 — Clean & Prepare | 9–17 | Organize columns, clean values, export a final clean dataset |
| Phase 3 — Analyze | 18–21 | Explore relationships and statistically test them |
| Phase 4 — Report | 22–23 | Define KPIs and charts for the final dashboard |

Each step below has its own markdown explanation followed by the code that performs it.

## **Environment Setup**

In [ ]:
# Load essential libraries for Data Handling (pandas, numpy), Plotting (matplotlib, seaborn), and Statistics (scipy)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Display plots directly inside the notebook
%matplotlib inline

# Set a default visual style for charts
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 50)

**Load the Dataset :**

In [ ]:
# Mount Google Drive to access the dataset stored in Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load the raw dataset into a Pandas DataFrame
RAW_PATH = "/content/drive/MyDrive/cityflo_bus_service_metro_cities.csv"
df = pd.read_csv(RAW_PATH)

# Display top 5 rows to verify data is loaded correctly
df.head()

,trip_id,booking_id,customer_id,customer_name,gender,age,city,route_id,route_name,origin_stop,destination_stop,bus_number,bus_type,driver_id,driver_name,trip_date,scheduled_departure,actual_departure,scheduled_arrival,actual_arrival,distance_km,fare_inr,discount_inr,payment_mode,booking_channel,seat_number,trip_status,cancellation_reason,rating,occupancy_pct,weather,is_peak_hour,subscription_type,device_type,gps_enabled,complaint_raised
0,TRP000768,BK47984879,CUST01278,Shalini Verma,Female,21.0,Mumbai,RTMU005,Lower Parel - Mulund,Lower Parel,Mulund,MH23CF4441,AC Sleeper,DRV0080,Aarav Bhat,2024-09-21,20:25,20:25,20:58,20:58,10.9,153,0.0,UPI,Corporate Portal,D8,Completed,NaN,5.0,84.6,Haze,0,Corporate Plan,iOS,true,No
1,TRP002439,BK14257322,CUST01290,Kalpana Chopra,Male,24.0,Hyderabad,RTHY038,HITEC City - Kondapur,HITEC City,Kondapur,TS27CF8651,AC Seater,DRV0068,Sanjay Joshi,2024-08-08,14:50,15:12,15:29,15:51,14.1,127,0.0,Debit Card,Kiosk,D7,Delayed-Completed,NaN,NaN,28.3,Heavy Rain,False,Weekly Pass,Android,True,0
2,TRP000793,BK67236768,CUST01285,Sai Naidu,Female,18.0,Pune,RTPU012,Aundh - Magarpatta,Aundh,Magarpatta,MH24CF3646,AC Seater,DRV0001,Lakshmi Patel,2024-03-19,08:15,08:15,09:42,09:42,27.6,234,19.0,Cash,Corporate Portal,D11,Completed,NaN,3.0,15.4,Cloudy,1,Single Ride,Android,Yes,No
3,TRP002802,BK59659578,CUST00625,Priya Patel,Female,23.0,Mumbai,RTMU010,Ghatkopar - Lower Parel,Ghatkopar,Lower Parel,MH13CF5853,AC Sleeper,DRV0036,Siddharth Kumar,2024-12-29,17:45,17:45,18:57,NaN,28.6,310,0.0,Debit Card,Corporate Portal,D11,No-show,Customer Request,NaN,68.1,Clear,No,Corporate Plan,Web,1,Yes
4,TRP001653,BK19231279,CUST00276,Shalini Kulkarni,F,20.0,Bangalore,RTBA024,Hebbal - Electronic City,Hebbal,Electronic City,KA22CF6238,AC Seater,DRV0045,Sunita Gupta,2024-05-15,13:00,13:40,13:35,14:15,17.8,202,0.0,UPI,Corporate Portal,A1,Delayed-Completed,NaN,NaN,23.7,Haze,No,Corporate Plan,iOS,True,0


## Phase 1 — Inspect the Raw Data (Steps 1–8)

Understand the shape, structure, and quality of the raw data.

### Step 1 — Total Number of Rows
Check the row count to understand the dataset's size (`df.shape[0]`).

In [ ]:
n_rows = df.shape[0]
print(f"Total rows: {n_rows}")

Total rows: 3258


### Step 2 — Total Number of Columns
Check the column count (`df.shape[1]`).

In [ ]:
n_cols = df.shape[1]
print(f"Total columns: {n_cols}")

Total columns: 36


### Step 3 — Understanding of Each Column
Go through every column and note what it represents, its expected values, and how it relates to the problem.

| Column | Represents | Expected values |
|---|---|---|
| trip_id | Unique trip identifier | `TRPxxxxxx` |
| booking_id | Unique booking transaction id | `BKxxxxxxxx` |
| customer_id | Unique customer identifier | `CUSTxxxxx` |
| customer_name | Passenger name | Free text |
| gender | Passenger gender | Male / Female / Other (messy variants) |
| age | Passenger age | 18–65 (a few data-entry errors) |
| city | Metro city of operation | Mumbai, Pune, Bangalore, Hyderabad, Chennai, Delhi NCR |
| route_id / route_name | Route identifier / "Origin - Destination" | e.g. `RTMU005`, "Lower Parel - Mulund" |
| origin_stop / destination_stop | Pickup / drop locality | Locality names per city |
| bus_number | Vehicle registration | State-coded reg. number |
| bus_type | Bus category | AC Seater, AC Sleeper, Non-AC Seater, Premium AC |
| driver_id / driver_name | Assigned driver | `DRVxxxx` / name |
| trip_date | Date of trip | Mixed formats (needs standardizing) |
| scheduled/actual departure & arrival | Timing | `HH:MM`, blank if cancelled |
| distance_km | Route distance | ~8–42 km |
| fare_inr | Ticket fare | Mixed formats (₹, INR, commas) |
| discount_inr | Discount applied | 0 or positive INR |
| payment_mode | Payment method | UPI, Card, Wallet, Cash, Net Banking |
| booking_channel | Booking source | Mobile App, Website, Kiosk, Corporate Portal |
| seat_number | Assigned seat | e.g. `A5` |
| trip_status | Trip outcome | Completed, Delayed-Completed, Cancelled, No-show |
| cancellation_reason | Why trip was cancelled/no-show | Populated only when relevant |
| rating | Customer rating | 1–5, missing for non-completed trips |
| occupancy_pct | Bus occupancy % | 0–100 (a few >100 data errors) |
| weather | Weather at trip time | Clear, Rain, Cloudy, Fog, Heavy Rain, Haze |
| is_peak_hour | Whether trip is in peak commute hours | Messy boolean (Yes/No/1/0/True/False) |
| subscription_type | Customer plan | Single Ride, Weekly/Monthly Pass, Corporate Plan |
| device_type | Booking device | Android, iOS, Web |
| gps_enabled | Whether GPS tracking was on | Messy boolean |
| complaint_raised | Whether customer complained | Messy boolean |


In [ ]:
# Create a structural summary table of every column
info_df = pd.DataFrame({
    "dtype": df.dtypes.astype(str),       # Data type of each column
    "n_unique": df.nunique(),             # Number of unique values per column
    "sample_value": df.iloc[0]            # First-row sample value for each column
})

# Convert the column names from the index into a normal column
info_df = info_df.reset_index().rename(columns={"index": "column"})
# Display table hiding the row numbers (index) using .style
info_df.style.hide(axis="index")

column,dtype,n_unique,sample_value
trip_id,object,3200,TRP000768
booking_id,object,3200,BK47984879
customer_id,object,1251,CUST01278
customer_name,object,866,Shalini Verma
gender,object,11,Female
age,float64,40,21.000000
city,object,12,Mumbai
route_id,object,60,RTMU005
route_name,object,60,Lower Parel - Mulund
origin_stop,object,40,Lower Parel


### Step 4 — Trim Extra Spaces
Strip leading/trailing whitespace from string columns and column headers — hidden spaces silently
break groupby, filtering, and joins. We first **detect** which columns are affected before fixing them
(the actual fix happens formally in Step 16, but we flag it here as required by Step 4).

In [ ]:
# Check for extra spaces in column names
bad_headers = [col for col in df.columns if col != col.strip()]
print("Bad headers:", bad_headers)

# Check for extra spaces inside text columns
for col in df.select_dtypes(include="object"):
    raw = df[col].astype(str)

    # Count rows where the value is different after removing spaces
    issue_count = (raw != raw.str.strip()).sum()

    # Display only columns that contain extra spaces
    if issue_count > 0:
        print(f"Column '{col}' has {issue_count} rows with extra spaces.")

Bad headers: []
Column 'customer_name' has 122 rows with extra spaces.
Column 'gender' has 783 rows with extra spaces.
Column 'city' has 104 rows with extra spaces.


### Step 5 — Remove Duplicate Elements
Identify and drop duplicate rows (`df.duplicated()`, `df.drop_duplicates()`).

In [ ]:
# Count the number of fully duplicated rows
n_dupes = df.duplicated().sum()
print(f"Fully duplicated rows: {n_dupes}")

# Show some duplicate rows for inspection
df.loc[df.duplicated(keep=False)].sort_values("trip_id").head(6)

Fully duplicated rows: 58


,trip_id,booking_id,customer_id,customer_name,gender,age,city,route_id,route_name,origin_stop,destination_stop,bus_number,bus_type,driver_id,driver_name,trip_date,scheduled_departure,actual_departure,scheduled_arrival,actual_arrival,distance_km,fare_inr,discount_inr,payment_mode,booking_channel,seat_number,trip_status,cancellation_reason,rating,occupancy_pct,weather,is_peak_hour,subscription_type,device_type,gps_enabled,complaint_raised
2082,TRP000045,BK31856387,CUST00757,Ananya Menon,M,32.0,Pune,RTPU014,Shivajinagar - Aundh,Shivajinagar,Aundh,MH48CF6442,Premium AC,DRV0010,Sai Iyer,2024-12-12,20:20,NaN,20:41,NaN,9.3,0,0.0,NaN,Corporate Portal,D1,Cancelled,Driver Unavailable,NaN,NaN,Haze,Yes,Monthly Pass,Android,True,0
2771,TRP000045,BK31856387,CUST00757,Ananya Menon,M,32.0,Pune,RTPU014,Shivajinagar - Aundh,Shivajinagar,Aundh,MH48CF6442,Premium AC,DRV0010,Sai Iyer,2024-12-12,20:20,NaN,20:41,NaN,9.3,0,0.0,NaN,Corporate Portal,D1,Cancelled,Driver Unavailable,NaN,NaN,Haze,Yes,Monthly Pass,Android,True,0
2765,TRP000085,BK64561490,CUST00511,Sunita Malhotra,Female,25.0,Mumbai,RTMU004,Thane West - BKC,Thane West,BKC,MH16CF7658,Premium AC,DRV0048,Divya Rao,2024-12-13,20:10,20:10,21:53,21:53,41.7,752,0.0,Debit Card,Corporate Portal,A7,Completed,NaN,5.0,95.0,Haze,False,Weekly Pass,iOS,true,0
1668,TRP000085,BK64561490,CUST00511,Sunita Malhotra,Female,25.0,Mumbai,RTMU004,Thane West - BKC,Thane West,BKC,MH16CF7658,Premium AC,DRV0048,Divya Rao,2024-12-13,20:10,20:10,21:53,21:53,41.7,752,0.0,Debit Card,Corporate Portal,A7,Completed,NaN,5.0,95.0,Haze,False,Weekly Pass,iOS,true,0
2093,TRP000234,BK53828083,CUST00340,Karan Pillai,Male,19.0,Pune,RTPU013,Hadapsar - Kothrud,Hadapsar,Kothrud,MH17CF2891,AC Seater,DRV0065,Vikram Bansal,06/01/2024,09:10,09:37,09:29,09:56,9.1,97,0.0,Cash,Mobile App,B15,Delayed-Completed,NaN,5.0,68.6,Fog,0,Corporate Plan,Web,1,No
2799,TRP000234,BK53828083,CUST00340,Karan Pillai,Male,19.0,Pune,RTPU013,Hadapsar - Kothrud,Hadapsar,Kothrud,MH17CF2891,AC Seater,DRV0065,Vikram Bansal,06/01/2024,09:10,09:37,09:29,09:56,9.1,97,0.0,Cash,Mobile App,B15,Delayed-Completed,NaN,5.0,68.6,Fog,0,Corporate Plan,Web,1,No


### Step 6 — Check Memory Size
Check memory usage (`df.memory_usage(deep=True)`) — flags if dtypes need downcasting for efficiency.

In [ ]:
# Calculate memory usage for each column
mem = df.memory_usage(deep=True)
print(mem)

# Calculate and display the total memory used by the dataset
print(f"\nTotal memory: {mem.sum() / 1024**2:.2f} MB")

Index                     132
trip_id                188964
booking_id             192222
customer_id            188964
customer_name          198257
gender                 173487
age                     26064
city                   183804
route_id               182448
route_name             227747
origin_stop            188905
destination_stop       188710
bus_number             192222
bus_type               193935
driver_id              182448
driver_name            196754
trip_date              193857
scheduled_departure    175932
actual_departure       167572
scheduled_arrival      175932
actual_arrival         160092
distance_km             26064
fare_inr               180112
discount_inr            26064
payment_mode           174607
booking_channel        190617
seat_number            167407
trip_status            191404
cancellation_reason    127439
rating                  26064
occupancy_pct           26064
weather                176941
is_peak_hour           168351
subscripti

### Step 7 — Check Data Type of Columns
Verify each column's dtype matches what it should be (e.g., dates not stored as text, numbers not
stored as objects).

In [ ]:
df.dtypes

,0
trip_id,object
booking_id,object
customer_id,object
customer_name,object
gender,object
age,float64
city,object
route_id,object
route_name,object
origin_stop,object


In [ ]:
# Define columns that should contain numeric values
should_be_numeric = [
    "age", "distance_km",
    "fare_inr", "discount_inr",
    "rating", "occupancy_pct"
]

# Define columns that should contain date values
should_be_datetime = ["trip_date"]

# Define columns that should contain True or False values
should_be_boolean = [
    "is_peak_hour", "gps_enabled",
    "complaint_raised"
]

# Check the current data types of numeric columns
print("Currently wrong dtype (numeric expected):")
print(df[should_be_numeric].dtypes)

# Check the current data type of the date column
print("\nCurrently wrong dtype (datetime expected):")
print(df[should_be_datetime].dtypes)

# Check the current data types of boolean columns
print("\nCurrently wrong dtype (boolean expected):")
print(df[should_be_boolean].dtypes)

Currently wrong dtype (numeric expected):
age              float64
distance_km      float64
fare_inr          object
discount_inr     float64
rating           float64
occupancy_pct    float64
dtype: object

Currently wrong dtype (datetime expected):
trip_date    object
dtype: object

Currently wrong dtype (boolean expected):
is_peak_hour        object
gps_enabled         object
complaint_raised    object
dtype: object


### Step 8 — Check Total Number of Null Values
Count missing values per column (`df.isnull().sum()`) to plan the cleaning strategy.

In [ ]:
# Count missing values in each column
null_counts = df.isnull().sum().sort_values(ascending=False)

# Calculate the percentage of missing values in each column
null_pct = (null_counts / len(df) * 100).round(2)

# Create a summary table and show only columns with missing values
pd.DataFrame({
    "nulls": null_counts,
    "pct_missing": null_pct
}).loc[null_counts > 0]

,nulls,pct_missing
cancellation_reason,2538,77.90
rating,1075,33.00
actual_arrival,720,22.10
actual_departure,380,11.66
payment_mode,380,11.66
occupancy_pct,380,11.66
discount_inr,184,5.65
age,64,1.96
driver_name,32,0.98


## Phase 2 — Clean & Prepare (Steps 9–17)

Organize columns, clean values, and export a final clean dataset.

### Step 9 — Split Columns: Numerical vs Categorical
Separate columns into numerical and categorical groups — they need different analysis and cleaning
approaches. (Split is based on *intended* type, since several numeric columns are still stored as
text at this point.)

In [ ]:
# List columns that contain numerical data
numerical_cols = [
    "age",     "distance_km",
    "fare_inr", "discount_inr",
    "rating", "occupancy_pct"
]

# List columns that contain categorical data
categorical_cols = [
    "gender", "city",
    "bus_type", "payment_mode",
    "booking_channel", "trip_status",
    "cancellation_reason", "weather",
    "subscription_type", "device_type",
    "is_peak_hour", "gps_enabled",
    "complaint_raised"
]

# List columns that contain date or time information
datetime_cols = [
    "trip_date", "scheduled_departure",
    "actual_departure", "scheduled_arrival",
    "actual_arrival"
]

# List columns that mainly work as identifiers
id_cols = [
    "trip_id", "booking_id",
    "customer_id", "route_id",
    "driver_id", "bus_number"
]

print("Numerical:", numerical_cols)
print("\nCategorical:", categorical_cols)
print("\nDate/Time:", datetime_cols)
print("\nID columns:", id_cols)

Numerical: ['age', 'distance_km', 'fare_inr', 'discount_inr', 'rating', 'occupancy_pct']

Categorical: ['gender', 'city', 'bus_type', 'payment_mode', 'booking_channel', 'trip_status', 'cancellation_reason', 'weather', 'subscription_type', 'device_type', 'is_peak_hour', 'gps_enabled', 'complaint_raised']

Date/Time: ['trip_date', 'scheduled_departure', 'actual_departure', 'scheduled_arrival', 'actual_arrival']

ID columns: ['trip_id', 'booking_id', 'customer_id', 'route_id', 'driver_id', 'bus_number']


### Step 10 — `describe()` of Numerical Columns
Run `df.describe()` to see count, mean, std, min, quartiles, and max. Note: `fare_inr` still has
mixed text formatting at this stage, so we coerce it to numeric first just for this preview
(the permanent fix happens in Step 16).

In [ ]:
# Create a temporary copy so the original dataset is not changed
preview = df.copy()

# Remove currency symbols, text, commas, and extra spaces from fare values
preview["fare_inr_preview"] = (
    preview["fare_inr"].astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("INR", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

# Convert the cleaned fare values into numeric format
# Invalid values will become missing values
preview["fare_inr_preview"] = pd.to_numeric(
    preview["fare_inr_preview"],
    errors="coerce"
)

# Check summary statistics before applying the cleaning to the main dataset
preview[
    [
        "age",
        "distance_km",
        "fare_inr_preview",
        "discount_inr",
        "rating",
        "occupancy_pct"
    ]
].describe()

,age,distance_km,fare_inr_preview,discount_inr,rating,occupancy_pct
count,3194.000000,3258.000000,3258.000000,3074.000000,2183.000000,2878.000000
mean,29.477145,24.696071,287.565991,14.365973,3.798443,58.313343
std,8.251467,10.311346,223.933495,31.070065,1.163396,25.073318
min,-5.000000,8.300000,0.000000,0.000000,1.000000,15.100000
25%,23.000000,16.000000,132.250000,0.000000,3.000000,36.700000
50%,29.000000,25.700000,286.000000,0.000000,4.000000,58.500000
75%,35.000000,33.100000,402.000000,8.000000,5.000000,79.500000
max,130.000000,41.800000,3288.000000,203.000000,5.000000,138.800000


### Step 11 — Write a Summary of the Dataset

**Summary:** This is a synthetic, trip-level dataset simulating a premium AC bus service
(CityFlo-style) operating across six Indian metro cities — Mumbai, Pune, Bangalore, Hyderabad,
Chennai, and Delhi NCR. Each of the 3,258 rows represents one passenger trip booked across 2024,
capturing customer demographics, route/bus details, scheduled vs. actual timings, fare and payment
information, occupancy, ratings, and trip outcomes (completed, delayed, cancelled, or no-show).

**Source:** Synthetically generated for EDA-workflow practice (not real operator data).

**Size:** 3,258 rows × 36 columns (~882 KB), including 58 intentional exact-duplicate rows.

**General quality:** Deliberately "raw" — it contains inconsistent category labels
(e.g. `Male`/`M`/`male`), mixed boolean representations (`Yes`/`No`/`1`/`0`/`True`/`False`), mixed
date formats, currency-symbol/comma inconsistencies in `fare_inr`, stray whitespace, missing values
in several columns (mostly context-dependent, e.g. no rating for cancelled trips), and a handful of
outliers / data-entry errors (negative ages, >100% occupancy, extreme fares).

In [ ]:
# Display the total number of rows and columns
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")

# Display all unique cities after removing extra spaces
print(
    f"Cities covered: {sorted(df['city'].str.strip().unique())}"
)

# Check the minimum and maximum date values before date conversion
print(
    f"Date range (raw, unparsed sample): "
    f"{df['trip_date'].min()} .. {df['trip_date'].max()}"
)

# Display the number of trips in each trip status
print(
    f"Trip status breakdown:\n"
    f"{df['trip_status'].value_counts()}"
)

Rows: 3,258  |  Columns: 36
Cities covered: ['Bangalore', 'Chennai', 'Delhi NCR', 'Hyderabad', 'Mumbai', 'Pune']
Date range (raw, unparsed sample): 01-Feb-2024 .. September 27, 2024
Trip status breakdown:
trip_status
Completed            2148
Delayed-Completed     390
Cancelled             380
No-show               340
Name: count, dtype: int64


### Step 12 — Write the Problem Statement

**Problem Statement:** *What factors drive trip delays, cancellations, and low customer
satisfaction in CityFlo's metro-city bus operations, and how do demand, revenue, and occupancy vary
across cities, routes, bus types, and time (peak vs. non-peak, day of week, month)?*

This drives every scoping decision from here on — which columns are required, which derived metrics
matter, and which relationships we test statistically.

### Step 13 — Mention the Columns That Are Required
Columns needed to answer the problem statement above:

- **Identifiers (for joins/grouping):** `trip_id`, `customer_id`, `route_id`
- **Where:** `city`, `route_name`, `origin_stop`, `destination_stop`, `bus_type`
- **When:** `trip_date`, `scheduled_departure`, `actual_departure`, `scheduled_arrival`, `actual_arrival`, `is_peak_hour`
- **Trip outcome:** `trip_status`, `cancellation_reason`, `rating`, `occupancy_pct`, `complaint_raised`
- **Money:** `distance_km`, `fare_inr`, `discount_inr`, `payment_mode`
- **Context:** `weather`, `subscription_type`, `booking_channel`, `gps_enabled`
- **Demographics:** `gender`, `age`

### Step 14 — Drop the Columns That Are Not Required
Not needed to answer the problem statement — either purely transactional IDs with no analytic
value, or personally identifiable information not required once `customer_id` is retained for
grouping:

- `booking_id` — transactional ID, no analytic value once `trip_id` exists
- `customer_name` — PII, not needed for aggregate analysis (`customer_id` retained)
- `driver_id`, `driver_name` — outside the stated problem's scope (route/city/ops level, not driver-level)
- `bus_number` — redundant once `bus_type` + `city` are retained
- `seat_number` — no analytic value
- `device_type` — outside the stated problem's scope

In [ ]:
# List columns that are not needed for further analysis
cols_to_drop = [
    "booking_id", "customer_name",
    "driver_id", "driver_name",
    "bus_number", "seat_number",
    "device_type"
]

# Remove the unnecessary columns
df = df.drop(columns=cols_to_drop)

# Display how many columns were removed
print(
    f"Dropped {len(cols_to_drop)} columns. "
    f"Remaining: {df.shape[1]}"
)

# Display the remaining column names
df.columns.tolist()

Dropped 7 columns. Remaining: 29


['trip_id',
 'customer_id',
 'gender',
 'age',
 'city',
 'route_id',
 'route_name',
 'origin_stop',
 'destination_stop',
 'bus_type',
 'trip_date',
 'scheduled_departure',
 'actual_departure',
 'scheduled_arrival',
 'actual_arrival',
 'distance_km',
 'fare_inr',
 'discount_inr',
 'payment_mode',
 'booking_channel',
 'trip_status',
 'cancellation_reason',
 'rating',
 'occupancy_pct',
 'weather',
 'is_peak_hour',
 'subscription_type',
 'gps_enabled',
 'complaint_raised']

### Step 15 — Add Derived Columns If Required
Useful columns not present in the raw data, created from existing ones:

- `trip_year`, `trip_month`, `trip_weekday` — extracted from `trip_date` once parsed
- `delay_minutes` — actual vs. scheduled departure gap
- `is_delayed` — flag when `delay_minutes > 5`
- `net_revenue_inr` — `fare_inr` minus `discount_inr`
- `age_group` — binned age brackets

(Date parsing and numeric coercion happen properly in Step 16 — derived columns below use the
cleaned versions produced there, computed together to avoid redundant parsing.)

### Step 16 — Perform Cleaning Operations on Each Column
Applying the right cleaning method based on each column's type and role, as outlined in the
checklist: text/categorical standardization, numerical imputation & outlier handling, date/time
parsing, ID validation, boolean normalization, and cross-column consistency checks.

**16a — Text / Categorical columns:** standardize case, strip whitespace, fix inconsistent
labels (e.g. gender variants), normalize boolean-style flag columns to a single representation.

In [ ]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Select all text columns
str_cols = df.select_dtypes(include="object").columns

# Clean extra spaces from all text values
for c in str_cols:
    df[c] = df[c].astype(str).str.strip()

    # Convert empty or invalid text values into missing values
    df.loc[df[c].isin(["", "nan", "None"]), c] = np.nan

# Fix inconsistent gender labels -> Male / Female / Other
gender_map = {
    "male": "Male", "m": "Male", "male ": "Male",
    "female": "Female", "f": "Female",
    "other": "Other", "o": "Other",
}
df["gender"] = df["gender"].str.lower().map(gender_map).fillna(df["gender"])
print(df["gender"].value_counts())

gender
Female    1655
Male      1455
Other      148
Name: count, dtype: int64


In [ ]:
# Create a mapping for different True and False values
bool_map = {
    "yes": True, "y": True, "1": True, "true": True,
    "no": False, "n": False, "0": False, "false": False
}

# Select columns that should contain boolean values
cols = [
    "is_peak_hour", "gps_enabled", "complaint_raised"
]

# Convert all possible values into True or False
for c in cols:

    # Clean the text before applying the mapping
    cleaned = df[c].astype(str).str.strip().str.lower()

    # Convert values using the boolean mapping and Unknown values will become missing values
    df[c] = cleaned.map(bool_map)

# Check the final values in each boolean column
df[cols].apply(
    lambda s: s.value_counts(dropna=False)
)

,is_peak_hour,gps_enabled,complaint_raised
False,1957,318,3047
True,1301,2940,211


**16b — Numerical columns:** type-correct `fare_inr` (strip ₹ / INR / commas), handle missing
values, detect & treat outliers (IQR method), check skewness/kurtosis, and fix impossible values
(negative ages, >100% occupancy).

In [ ]:
# Clean the fare column by removing currency symbols and commas
df["fare_inr"] = (
    df["fare_inr"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("INR", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

# Convert fare values into numeric format
# Invalid values will become missing values
df["fare_inr"] = pd.to_numeric(
    df["fare_inr"],
    errors="coerce"
)

# Convert other numerical columns into numeric format
for c in [
    "age", "distance_km",
    "discount_inr", "rating",
    "occupancy_pct"
]:

    # Invalid values will become missing values
    df[c] = pd.to_numeric(
        df[c],
        errors="coerce"
    )

# Check the final data types
df[
    [
        "age", "distance_km",
        "fare_inr", "discount_inr",
        "rating", "occupancy_pct"
    ]
].dtypes

,0
age,float64
distance_km,float64
fare_inr,int64
discount_inr,float64
rating,float64
occupancy_pct,float64


In [ ]:
# Replace unrealistic age values with missing values
df.loc[
    (df["age"] < 5) | (df["age"] > 100), "age"
] = np.nan

# Replace occupancy values outside 0 to 100 with missing values
df.loc[
    (df["occupancy_pct"] < 0) | (df["occupancy_pct"] > 100), "occupancy_pct"
] = np.nan

# Replace zero or negative fares with missing values
df.loc[
    df["fare_inr"] <= 0, "fare_inr"
] = np.nan

# Calculate the first and third quartiles of fare
q1, q3 = df["fare_inr"].quantile([0.25, 0.75])

# Calculate the first and third quartiles of fare
iqr = q3 - q1

# Calculate the lower and upper bounds for outlier detection
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

# Identify rows containing fare outliers
outliers = df[
    (df["fare_inr"] < lower) | (df["fare_inr"] > upper)
]

print(f"fare_inr IQR bounds: [{lower:.0f}, {upper:.0f}]  |  Outliers flagged: {len(outliers)}")

# Cap extreme fare values instead of deleting entire rows
# This keeps the trip records while limiting extreme values
df["fare_inr"] = df["fare_inr"].clip(
    lower=lower, upper=upper
)

fare_inr IQR bounds: [-158, 766]  |  Outliers flagged: 30


In [ ]:
# Check the distribution shape of important numerical columns
for c in [
    "age", "distance_km",
    "fare_inr", "occupancy_pct"
]:

    # Print skewness and kurtosis for each column
    print(f"{c:15s} skew={df[c].skew():+.2f}   kurtosis={df[c].kurt():+.2f}")

age             skew=+0.36   kurtosis=-0.43
distance_km     skew=-0.13   kurtosis=-1.14
fare_inr        skew=+0.46   kurtosis=-0.22
occupancy_pct   skew=-0.01   kurtosis=-1.22


In [ ]:
# Fill missing age values with the median age
df["age"] = df["age"].fillna(
    df["age"].median()
)
df["discount_inr"] = df["discount_inr"].fillna(0)

# Fill missing occupancy values using the median occupancy
# within each trip status group
df["occupancy_pct"] = (
    df.groupby("trip_status")["occupancy_pct"]
    .transform(
        lambda s: s.fillna(s.median())
    )
)

**16c — Date / Time columns:** standardize `trip_date` (mixed formats) with `pd.to_datetime`,
then extract derived parts.

In [ ]:
# Convert trip_date into datetime format
# Invalid date values will become missing values
df["trip_date"] = pd.to_datetime(
    df["trip_date"], format="mixed",
    dayfirst=True, errors="coerce"
)

# Extract the year from the trip date
df["trip_year"] = df["trip_date"].dt.year

# Extract the month number from the trip date
df["trip_month"] = df["trip_date"].dt.month

# Extract the weekday name from the trip date
df["trip_weekday"] = df["trip_date"].dt.day_name()

# Count dates that could not be converted
print(f"Unparseable trip_date rows: {df['trip_date'].isna().sum()}")

# Preview the newly created date features
df[
    ["trip_date", "trip_year",
     "trip_month", "trip_weekday"]
].head()

Unparseable trip_date rows: 0


,trip_date,trip_year,trip_month,trip_weekday
0,2024-09-21,2024,9,Saturday
1,2024-08-08,2024,8,Thursday
2,2024-03-19,2024,3,Tuesday
3,2024-12-29,2024,12,Sunday
4,2024-05-15,2024,5,Wednesday


In [ ]:
# Convert time values from HH:MM format into total minutes
def to_minutes(t):
    # Return missing value if the time is missing
    if (pd.isna(t) or t == "" or str(t).lower() == "nan"):
        return np.nan

    # Split hours and minutes and Convert the time into total minutes
    h, m = str(t).split(":")
    return int(h) * 60 + int(m)

# Convert scheduled departure and actual departure time into minutes
sched_dep_min = df["scheduled_departure"].apply(to_minutes)
actual_dep_min = df["actual_departure"].apply(to_minutes)

# Calculate departure delay in minutes
delay = actual_dep_min - sched_dep_min

# Handle trips that cross midnight
delay = delay.apply(
    lambda d: d + 1440
    if pd.notna(d) and d < -600
    else d
)

delay = delay.apply(
    lambda d: d - 1440
    if pd.notna(d) and d > 600
    else d
)

# Save the calculated delay in the dataset
df["delay_minutes"] = delay

# Mark trips delayed by more than 5 minutes
df["is_delayed"] = df["delay_minutes"] > 5

# Preview the new delay columns
df[
    ["scheduled_departure", "actual_departure",
     "delay_minutes", "is_delayed"]
].head()

,scheduled_departure,actual_departure,delay_minutes,is_delayed
0,20:25,20:25,0.0,False
1,14:50,15:12,22.0,True
2,08:15,08:15,0.0,False
3,17:45,17:45,0.0,False
4,13:00,13:40,40.0,True


In [ ]:
# Calculate net revenue after subtracting discount
df["net_revenue_inr"] = (
    df["fare_inr"] - df["discount_inr"]
).clip(lower=0)

# Divide passengers into age groups
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 45, 60, 100],
    labels=["18-25", "26-35", "36-45", "46-60", "60+"]
)

# Preview the newly created columns
df[["net_revenue_inr", "age_group"]].head()

,net_revenue_inr,age_group
0,153.0,18-25
1,127.0,18-25
2,215.0,18-25
3,310.0,18-25
4,202.0,18-25


**16d — ID / unique-key columns:** validate uniqueness and format consistency.

In [ ]:
# Check whether every trip ID follows the expected format
print(
    "trip_id unique format check (all match TRP######):",
     df["trip_id"]
     .str.match(r"^TRP\d{6}$")
     .all()
)

# Check whether every customer ID follows the expected format
print(
    "customer_id unique format check (all match CUST#####):",
     df["customer_id"]
     .str.match(r"^CUST\d{5}$")
     .all()
)

# Check whether every route ID follows the expected format
print(
     "route_id unique format check (all match RT..###):",
      df["route_id"]
      .str.match(r"^RT[A-Z]{2}\d{3}$")
      .all()
)

trip_id unique format check (all match TRP######): True
customer_id unique format check (all match CUST#####): True
route_id unique format check (all match RT..###): True


**16e — Cross-column & general checks:** logical consistency (arrival not before departure),
referential checks (cancellation_reason only present for Cancelled/No-show trips), remove the
already-identified duplicates, and reset the index.

In [ ]:
# Save the row count before removing duplicates
before = len(df)

# Keep only the first occurrence of each trip ID
df = df.drop_duplicates(
    subset="trip_id", keep="first"
)
# Display how many duplicate rows were removed
print(f"Dropped {before - len(df)} duplicate rows")

# Check if cancellation reasons exist for trips
# that are not Cancelled or No-show
bad_reason = df[
    (df["cancellation_reason"].notna()) &
    (~df["trip_status"].isin(["Cancelled", "No-show"]))
]
print(f"Rows with cancellation_reason but status not Cancelled/No-show: {len(bad_reason)}")

# Check cancelled trips that still have delay values
mismatch = df[
    (df["trip_status"] == "Cancelled") &
     (df["delay_minutes"].notna())
]
print(f"Cancelled trips with a non-null delay_minutes (should be 0): {len(mismatch)}")

# Reset the row index after cleaning
df = df.reset_index(drop=True)

# Display the final dataset shape
print(f"\nFinal cleaned shape: {df.shape}")

Dropped 58 duplicate rows
Rows with cancellation_reason but status not Cancelled/No-show: 0
Cancelled trips with a non-null delay_minutes (should be 0): 0

Final cleaned shape: (3200, 36)


### Step 17 — Convert Into Final `cleaned.csv` File
Export the cleaned, prepared dataset as a single `cleaned.csv` to use as the base for all further
analysis.

In [ ]:
# Define the file name for the cleaned dataset
CLEANED_PATH = "cityflo_bus_service_metro_cities_cleaned.csv"

# Save the cleaned dataset as a CSV file
df.to_csv(CLEANED_PATH, index=False)

print(f"Saved cleaned dataset -> {CLEANED_PATH}  |  shape={df.shape}")

Saved cleaned dataset -> cityflo_bus_service_metro_cities_cleaned.csv  |  shape=(3200, 36)
